In [0]:
%sql
SELECT *
FROM main.default.sales_v3

id,date,sale,city
100,2026-01-31,4002,Suva
101,2026-01-26,5983,Alor Star
102,2026-05-04,6643,Adak
103,2026-01-29,5027,Palikir
104,2026-04-15,3667,Mexicali
105,2026-04-01,5517,Alofi
106,2026-04-27,8122,Anadyr (town)
107,2026-01-11,5903,Kaliningrad
108,2026-03-30,8063,Lahore
109,2026-04-14,6885,Rome


Testing to make sure the v3 tables has the correct dates in this date column

In [0]:
%sql
select min(date), max(date)
from main.default.sales_v3

min(date),max(date)
2026-01-01,2026-05-04


In [0]:
from datetime import date, timedelta, datetime

dbutils.widgets.dropdown("time_period", "weekly", ["weekly", "monthly", "daily"])
time_period = dbutils.widgets.get("time_period")

today = date.today()
# print(today) you can check your date functions and modules are imported correctly here

if time_period == 'weekly':
    start_date = today-timedelta(days = today.weekday(), weeks=1) - timedelta(days=1)
    end_date = start_date + timedelta(days=6)

elif time_period == 'monthly':
    first = today.replace(day=1)
    end_date = first - timedelta(days=1)
    start_date = first - timedelta(days=end_date.day)

else:
    start_date = today-timedelta(days=1)
    end_date = start_date

display(spark.sql(f"""
                  SELECT SUM(sale), city FROM main.default.sales_v3 s 
                  WHERE s.date BETWEEN '{start_date}' AND '{end_date}'
                  GROUP BY city """))

print(start_date, end_date)
# checking to make sure our date fields and drop downs are working correctly

SUM(sale),city
13190,Anadyr (town)
5509,Canberra
6940,Kaohsiung
4286,Ipoh
3606,Bucharest
2638,Las Palmas de Gran Canaria
3460,Barcelona
6929,Washington
2858,Ndola
3949,St. Louis


2026-04-26 2026-05-02


Writing an output to a Delta table that would represent what a potential client would want to see.

In [0]:
df = spark.sql(f"SELECT SUM(sale) AS total_sale, city FROM main.default.sales_v3 WHERE date BETWEEN '{start_date}' AND '{end_date}' GROUP BY city")
df.write.format("delta").mode("overwrite").saveAsTable("main.default.sales_report_output")

Total Sales report the week of 2026-04-26 to 2026-05-02

In [0]:
%sql
SELECT *
FROM main.default.sales_report_output

total_sale,city
9716,Malé
2509,Nukus
5301,Prague
3059,Maceió
3201,Moscow
2638,Las Palmas de Gran Canaria
7453,Amman
8790,Strasbourg
13190,Anadyr (town)
5509,Canberra
